### Notebook Structure
1. Introduction & Objective
2. Import Libraries
3. Load Trained Models & Scaler
4. Load NSL-KDD Dataset
5. NSL-KDD Preprocessing
6. Feature Alignment
7. Model Testing
8. Evaluation & Results
9. Discussion


## NSL-KDD Cross-Dataset Testing
This notebook evaluates the generalization performance of the trained
Random Forest and SGD-SVM models on the NSL-KDD dataset.

The models were trained on CICIDS2017 and are **not retrained** here.
Only preprocessing alignment and inference are performed.

In [2]:
# Importing libraries
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, classification_report

import joblib


In [3]:
# loading trained mdoels, onlu random forest and sgd-svm

rf_model = joblib.load("../models/random_forest_model.pkl")
svm_model = joblib.load("../models/sgd_svm_model.pkl")

print("RF expects features:", rf_model.n_features_in_)
print("SVM expects features:", svm_model.n_features_in_)


RF expects features: 78
SVM expects features: 78


In [4]:
#defining teh nsl-kdd column names
column_names = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files',
    'num_outbound_cmds','is_host_login','is_guest_login','count',
    'srv_count','serror_rate','srv_serror_rate','rerror_rate',
    'srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

In [5]:
nsl_df = pd.read_csv(
    "../datasets/raw/NSL-KDD/kdd_test.csv"
)

nsl_df.shape

(22544, 42)

In [6]:
nsl_df.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'labels'],
      dtype='object')

In [9]:
nsl_df['binary_label'] = nsl_df['labels'].apply(
    lambda x: 0 if x == 'normal' else 1
)

y_true = nsl_df['binary_label']
y_true.value_counts()


binary_label
1    11299
0    11245
Name: count, dtype: int64

In [11]:
# minimal preprocessing to evaluare generalization
# dropping categorical columns
X_nsl = nsl_df.drop(columns=[
    'protocol_type',
    'service',
    'flag',
    'labels',
    'binary_label'
])

In [12]:
# ensuring numeric features only
X_nsl = X_nsl.select_dtypes(include=['int64', 'float64'])
X_nsl.shape

(22544, 38)

In [13]:
# feature alignment
expected_features = rf_model.n_features_in_
X_nsl = X_nsl.iloc[:, :expected_features]

X_nsl.shape

(22544, 38)

In [16]:
X_nsl_np = X_nsl.to_numpy() # fixing the input format mismatch, during the fit numpy array was used and now its in data frames

In [17]:
# raw predictions
rf_raw_preds = rf_model.predict(X_nsl_np)
svm_raw_preds = svm_model.predict(X_nsl_np)

c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


ValueError: X has 38 features, but RandomForestClassifier is expecting 78 features as input.

### the classical cross-dataset validation couldn't be carried out due to feature mismatch, the NSL-KDD only has 38 features while the MOdel expects 78 features, and hence we will be performing the evaluation using the stratification, where only the unseen dataset i.e. test split will be used.

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [6]:
X = pd.read_csv("../datasets/feature_sets/cicids_combined_mixed_features.csv")
y = pd.read_csv("../datasets/feature_sets/cicids_combined_mixed_labels.csv")

In [5]:
print(type(y))
print(y.shape)

<class 'pandas.core.frame.DataFrame'>
(2827876, 1)


In [6]:
y = y.squeeze()

In [7]:
print(type(y))
print(y.shape)

<class 'pandas.core.series.Series'>
(2827876,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
rf_model = joblib.load("../models/random_forest_model.pkl")

X_test_rf = X_test[rf_model.feature_names_in_]

In [12]:
lr_model = joblib.load("../models/logistic_regression.pkl")

X_test_lr = X_test[rf_model.feature_names_in_]

In [13]:
svm_model = joblib.load("../models/sgd_svm_model.pkl")

X_test_svm = X_test[rf_model.feature_names_in_]

## Prediction and Evaluation

In [14]:
# random forest
y_pred_rf = rf_model.predict(X_test_rf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.9995261467954794
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    454265
           1       1.00      1.00      1.00     61711
           2       1.00      1.00      1.00     47209
           3       0.97      0.90      0.93      1230
           4       0.99      1.00      0.99      1159
           5       1.00      1.00      1.00         2

    accuracy                           1.00    565576
   macro avg       0.99      0.98      0.99    565576
weighted avg       1.00      1.00      1.00    565576



In [15]:
# logistic regression
y_pred_lr = lr_model.predict(X_test_lr)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.9707766949092607


c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

           0       0.97      0.99      0.98    454265
           1       0.96      0.95      0.96     61711
           2       0.94      0.83      0.88     47209
           3       0.89      0.74      0.81      1230
           4       0.89      0.82      0.86      1159
           5       0.00      0.00      0.00         2

    accuracy                           0.97    565576
   macro avg       0.78      0.72      0.75    565576
weighted avg       0.97      0.97      0.97    565576



c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [16]:
# sgd-svm 
y_pred_svm = svm_model.predict(X_test_svm)

print("SGD-SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

SGD-SVM Accuracy: 0.9532565031047994


c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

           0       0.95      0.99      0.97    454265
           1       0.94      0.93      0.94     61711
           2       0.97      0.65      0.78     47209
           3       0.82      0.51      0.63      1230
           4       0.75      0.15      0.25      1159
           5       0.00      0.00      0.00         2

    accuracy                           0.95    565576
   macro avg       0.74      0.54      0.59    565576
weighted avg       0.95      0.95      0.95    565576



c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
# isolation forest
iso_model = joblib.load("../models/isolation_forest_cicids_monday.pkl")
X_test_iso = X_test[iso_model.feature_names_in_]

In [9]:
y_pred_iso = iso_model.predict(X_test_iso) # prediction
y_pred_iso = np.where(y_pred_iso == 1, 0, 1) # binary conversion

In [10]:
# evaluation
print("Isolation Forest Accuracy:", accuracy_score(y_test, y_pred_iso))
print(classification_report(y_test, y_pred_iso))

Isolation Forest Accuracy: 0.7420081474461434


c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

           0       0.80      0.91      0.85    454265
           1       0.10      0.08      0.09     61711
           2       0.00      0.00      0.00     47209
           3       0.00      0.00      0.00      1230
           4       0.00      0.00      0.00      1159
           5       0.00      0.00      0.00         2

    accuracy                           0.74    565576
   macro avg       0.15      0.17      0.16    565576
weighted avg       0.65      0.74      0.70    565576



c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Kanchan Chaudhary\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
